# slv_productos
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Silver (nivel 2) · entidad Productos |
| Propósito | bronze.products_raw → silver.productos (SCD1). Por qué SCD1 : el precio histórico ya vive en cada ítem de pedido (precio_unitario). |
| Reglas | reglas_calidad.REGLAS["productos"] · tiene_descripcion importa para el RAG (nivel 5) |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 2.0 · 2026-10-12 |

## 1. Librerías y módulos comunes
- `dlt`: API de los pipelines declarativos (vistas, tablas, AUTO CDC, expectations).
- `comun` / `comun_gold` y `reglas_calidad`: utilidades y reglas compartidas por todas las entidades (archivos `.py` de la misma carpeta; el pipeline los encuentra gracias a `root_path`).

In [ ]:
import dlt
from pyspark.sql import functions as F

from comun import TECNICAS, con_secuencia, leer_bronze, parametros_cdc
from reglas_calidad import reglas

# AUTO CDC: nombre actual de la API; apply_changes es el nombre anterior.
aplicar_cdc = getattr(dlt, "create_auto_cdc_flow", None) or dlt.apply_changes

## 2. Vista de limpieza `limpio_productos`
Lee los cambios de bronze **como stream** (solo lo nuevo en cada corrida) y:
1. agrega el **orden de los cambios** (`_secuencia`: fecha efectiva, `_lsn`, `_seqval`);
2. **renombra** a snake_case en español y **normaliza** valores;
3. aplica las **reglas de calidad** en tres niveles: `fallar` (detiene el pipeline), `descartar` (no entra; la recoge la cuarentena) y `advertir` (entra y queda la métrica).

In [ ]:
@dlt.view(name="limpio_productos", comment="Productos de bronze renombrados y validados.")
@dlt.expect_all_or_fail(reglas("productos", "fallar"))      # rompe el modelo → detiene el pipeline
@dlt.expect_all_or_drop(reglas("productos", "descartar"))   # inválido → no entra (va a cuarentena)
@dlt.expect_all(reglas("productos", "advertir"))            # mejorable → entra y queda la métrica
def limpio_productos():
    df = con_secuencia(leer_bronze("products_raw"))
    return df.select(
        F.col("ProductID").alias("id_producto"),
        F.trim("SKU").alias("sku"),
        F.trim("ProductName").alias("nombre_producto"),
        F.trim("Category").alias("categoria"),
        F.col("Price").alias("precio"),
        F.col("Description").alias("descripcion"),
        F.lower("Status").alias("estado"),
        F.col("CreatedAt").alias("creado_en"),
        F.col("UpdatedAt").alias("actualizado_en"),
        F.col("Description").isNotNull().alias("tiene_descripcion"),
        *TECNICAS)

## 3. Tabla(s) silver con AUTO CDC: `productos`
AUTO CDC aplica los cambios **en su orden real** (`_secuencia`), se queda con el último por clave (resuelve duplicados y llegadas fuera de orden) y convierte los `D` de la fuente en **borrados**.
- **SCD1** → estado actual (una fila por clave).

In [ ]:
dlt.create_streaming_table(name="productos", comment="Productos: estado actual (SCD1) desde CDC de Azure SQL.")
aplicar_cdc(target="productos", source="limpio_productos", **parametros_cdc("id_producto"))
# Lo que no cumple una regla de descarte lo recoge slv_cuarentena.py.